In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Install and import required packages

In [ ]:
!pip install itables
!pip install -U sentence-transformers
!pip install pyvis
!pip install networkx==2.6.3

In [ ]:
%%capture
import pandas
import numpy
import re
import seaborn as sns
import matplotlib.pyplot as plt
from itables import show
from sentence_transformers import SentenceTransformer, util
import warnings
from pyvis.network import Network
import networkx as nx

warnings.filterwarnings('ignore')

model = SentenceTransformer('distilbert-base-nli-mean-tokens')
sns.set_theme()

In [ ]:
df = pandas.read_csv('/kaggle/input/tv-and-movie-metadata-with-genres-and-ratings-imbd/IMBD.csv')
input_df = df.copy()
input_df.info()

# Lets understand the data
![](https://cdn.chimpify.net/5f896ecda8587281208b456f/2021/05/mmt-SoM-datenSindDasNeueWasser-1200x627-blog-1.png)

## Clean the data to set the columns with relevant datatypes
Look for irregularities in the datatypes and fix it.<br>
As other notebooks covered some interesting visualizations, skipping those..

In [ ]:
# formatting
input_df['runtime'].fillna('0 min', inplace=True)
input_df['runtime'] = input_df['runtime'].apply(lambda x: x.split(' ')[0])
input_df['runtime'] = input_df['runtime'].apply(lambda x: x.replace(",", "")).astype(int)
input_df.loc[input_df['runtime'] == 0, 'runtime'] = pandas.NA

input_df['votes'].fillna('0', inplace=True)
input_df['votes'] = input_df['votes'].apply(lambda x: re.sub(r'[^0-9]', "", x)).astype(int)
input_df.loc[input_df['votes'] == 0, 'votes'] = pandas.NA

## Visualization -  Data distribution for rating and runtime fields

In [ ]:
plt.hist(input_df['rating'])
plt.grid(True)
plt.title("Rating distribution")
plt.show()

plt.hist(input_df['runtime'], range=[10, 300])
plt.grid(True)
plt.title("Runtime distribution")
plt.show()

## Data visualization - Data distribution of Votes field - pre and post binning

The curve for votes is highly narrow where as the min and max votes range is very high 

In [ ]:
votes = input_df.loc[input_df["votes"] > 0, "votes"]
plt.hist(input_df["votes"])
# sns.displot(input_df["votes"], kde=True)
plt.grid(True)
plt.title("Votes distribution")
plt.show()

bins = numpy.linspace(1, 100000)
digitized = numpy.digitize(input_df["votes"], bins)
bin_means = [input_df["votes"][digitized == i].mean() for i in range(1, len(bins))]
plt.hist(digitized)
plt.grid(True)
plt.title("Votes distribution post binning")
plt.show()

## Visualization - Votes vs Rating distribution

In [ ]:
sns.set_theme(style="ticks")
sketch = sns.JointGrid(x=input_df.loc[input_df["votes"] > 0, "rating"], y=votes)
sketch.ax_joint.set(yscale="log")
cax = sketch.figure.add_axes([.15, .55, .02, .2])
sketch.plot_joint(
    sns.histplot, discrete=(True, False),
    cmap="light:#03012d", pmax=.8, cbar=True, cbar_ax=cax
)
sketch.plot_marginals(sns.histplot, element="step", color="#03012d")
# plt.title("rating vs votes")
plt.show()

## Understanding Certificate vs Rating distribution
Plot shows that in general films/shows rated 7+ had better rating whereas R rated items had very high standard deviation

In [ ]:
fig, ax = plt.subplots(figsize=(25, 12))
sns.boxplot(x=input_df["certificate"], y=input_df["rating"])
plt.title("certification vs rating")
plt.show()

In [ ]:
sns.set_theme()
sns.displot(input_df, x="rating", hue="certificate", kind="kde", multiple="stack")
plt.title("Rating distribution across certificate")
plt.show()

## UI for simpler queries like searching and sorting the dataframe

In [ ]:
show(input_df)

In [ ]:
#referenced from https://www.kaggle.com/code/rajatraj0502/tv-movie-metadata-with-genres-and-ratings-2023
director_counts = input_df['director'].value_counts()
total_votes_by_director = df.groupby('director')['votes'].sum()
average_rating_by_director = input_df.groupby('director')['rating'].mean()
directors_with_more_than_10 = director_counts[director_counts > 10].index
filtered_total_votes_by_director = total_votes_by_director[total_votes_by_director.index.isin(directors_with_more_than_10)]
filtered_average_rating_by_director = average_rating_by_director[average_rating_by_director.index.isin(directors_with_more_than_10)]
directors_with_more_than_10 = director_counts[director_counts > 10].index
total_votes_by_director = input_df.groupby('director')['votes'].sum()
filtered_total_votes_by_director = total_votes_by_director[total_votes_by_director.index.isin(directors_with_more_than_10)]
top_50_directors = filtered_total_votes_by_director.nlargest(50).index

# Filter the average ratings and total votes for these directors
filtered_average_rating_by_director_top_50 = filtered_average_rating_by_director[filtered_average_rating_by_director.index.isin(top_50_directors)]
filtered_total_votes_by_director_top_50 = filtered_total_votes_by_director[filtered_total_votes_by_director.index.isin(top_50_directors)]

# Create a scatter plot for the average rating and total votes of these directors
plt.figure(figsize=(25, 15))
sns.set_theme()
sns.scatterplot(x=filtered_average_rating_by_director_top_50, y=filtered_total_votes_by_director_top_50, s=100)
plt.title('Average Rating vs Total Votes for Top 50 Directors')
plt.xlabel('Average Rating')
plt.ylabel('Total Votes')

# Annotate the directors' names
for line in range(0, filtered_total_votes_by_director_top_50.shape[0]):
     plt.text(filtered_average_rating_by_director_top_50[line], filtered_total_votes_by_director_top_50[line], 
     filtered_total_votes_by_director_top_50.index[line], horizontalalignment='left', 
     size='small', color='black')

plt.show()

## Clean the genre field for further processing

In [ ]:
genre = []
input_df['genre'] = input_df['genre'].apply(lambda x: x.replace(" ", "").split(","))

for row in range(0, len(input_df)):
    genre_cur = input_df.loc[row, 'genre']
    for val in genre_cur: genre.append(val)
        
genre = set(genre)
for val in genre: input_df['genre_' + val] = int(0)

for row in range(0, len(input_df)):
    genre_cur = input_df.loc[row, 'genre']
    for val in genre_cur: input_df.loc[row, 'genre_' + val] = 1

In [ ]:
def clean_stars(param):
    director = param['director']
    stars = param['stars']
    if not pandas.isnull(stars):
        stars = stars.replace(" ', '", "")
        stars = re.sub(r'[\[\]\']', "", stars)
        stars = stars.replace(", ", "")
        stars = stars.split(",")
    if not pandas.isnull(director):
        director = re.sub(r'[\[\]\']', "", director)
    return director, stars

input_df[['director', 'stars']] = input_df[['director', 'stars']].apply(clean_stars, axis=1, result_type='broadcast')

In [ ]:
dir_star_combo = {}
tmp_df = input_df[~input_df['director'].isna()]
tmp_df.reset_index(drop=True, inplace=True)
tmp_df = tmp_df[~tmp_df['stars'].isna()]
tmp_df.reset_index(drop=True, inplace=True)

In [ ]:
dir_star_combo = {}

for row in range(len(tmp_df)):
#     print(dir_star_combo)
    cur_dir = tmp_df.loc[row, 'director']
    cur_stars = tmp_df.loc[row, 'stars']
    
    if cur_dir in dir_star_combo:
        vals = dir_star_combo.get(cur_dir)
        for cur_star in cur_stars:
            if cur_star in vals: vals[cur_star] = vals.get(cur_star) + 1
            else: vals[cur_star] = 1
            dir_star_combo[cur_dir] = vals
    else:
        vals = {}
        for cur_star in cur_stars: vals[cur_star] = 1
        dir_star_combo[cur_dir] = vals 

In [ ]:
%%capture
import copy
dir_star_combo_org = copy.deepcopy(dir_star_combo)

for dir in list(dir_star_combo):
    vals = dir_star_combo.get(dir)
    for star in list(vals):
        if vals.get(star) < 3: del vals[star]
            
{k: v for k, v in dir_star_combo.items() if v}

In [ ]:
dir_star_df = pandas.DataFrame(columns=['director', 'star', 'collaberations'])

idx = 0
for dir in list(dir_star_combo):
    vals = dir_star_combo.get(dir)
    for star in list(vals):
        dir_star_df.loc[idx] = [dir, star, vals.get(star)]
        idx += 1

In [ ]:
# net = Network(notebook=True,)

# dir_star_net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white")
# dir_star_net.barnes_hut()

# sources = dir_star_df['director']
# targets = dir_star_df['star']
# weights = dir_star_df['collaberations']

# edge_data = zip(sources, targets, weights)

# for e in edge_data:
#     src = e[0]
#     dst = e[1]
#     w = e[2]

#     dir_star_net.add_node(src, src, title=src)
#     dir_star_net.add_node(dst, dst, title=dst)
#     dir_star_net.add_edge(src, dst, value=w)

# neighbor_map = dir_star_net.get_adj_list()

# # add neighbor data to node hover data
# for node in dir_star_net.nodes:
#     node["title"] += " Neighbors:<br>" + "<br>".join(neighbor_map[node["id"]])
#     node["value"] = len(neighbor_map[node["id"]])

# dir_star_net.show("collabs.html")

## Network graph to visualize the Director and Star combo who worked together for more than 3 projects
Showing only part of the graph (first 250 entries to avoid slowness of the notebook)

In [ ]:
sns.set_theme(style="ticks")
G = nx.from_pandas_edgelist(dir_star_df.loc[:250, :], 
                            source='director', 
                            target='star', 
                            edge_attr='collaberations')

net = Network(notebook=True, width=1000, height=600)
net.from_nx(G)
net.show("collabs.html")

## Content based recommender system which suggests movie/shows based on Genre and description

In [ ]:
%%capture
def calculate_similarity(df, usr_input):
    usr_input = usr_input.values
    df = df.values
    encoded_input = model.encode(usr_input[6])
    encoded = model.encode(df[6])
    
    cos_sim_desc = util.pytorch_cos_sim(encoded_input, encoded)
    cos_sim_genre = util.pytorch_cos_sim(usr_input[10:37].astype(numpy.float32), df[10:37].astype(numpy.float32))
    return cos_sim_genre.item(), cos_sim_desc.item()

# Assumes the movie watched is the first matching entry
def recommend_movie(input):
    tmp_df = input_df.copy()
    tmp_df = tmp_df.iloc[:100, :]
    index = tmp_df.index[tmp_df['movie'] == input].tolist()
    if len(index) < 1: print("movie name not found in the database")
    else:
        input_row = tmp_df.loc[index[0], :]
        tmp_df.drop(index[0], inplace=True)
        tmp_df[['genre_score', 'description_score']] = tmp_df.apply(calculate_similarity, args=(input_row,), axis=1, result_type='expand')
        tmp_df['consolidated_score'] = tmp_df.apply(lambda x: 0.65*x.genre_score + 0.35*x.description_score, axis=1)
        tmp_df.sort_values(by=['consolidated_score'], ascending=False, inplace=True)
        return input, tmp_df
    
usr_input, suggested = recommend_movie("The Witcher")

In [ ]:
print(f"the suggestions based on \"{usr_input}\" are:\n {suggested.iloc[:5, 0].values}")